# Hospital Readmission Risk Prediction for Diabetic Patients

**Author:** Abdifatah Hussein

---

## Phase 2 — Model Selection

Using preprocessed train/test splits, we benchmark three core classifiers (Random Forest, XGBoost, MLP) across three class-balancing strategies, then probe ensemble methods. The champion will carry forward to the tuning phase.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('..')
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import (
    RandomForestClassifier, VotingClassifier,
    BaggingClassifier, StackingClassifier
)

from src.preprocessing.helpers import describe_dataset
from src.evaluation import *

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({'figure.figsize': (11, 6), 'font.size': 14})
%config InlineBackend.figure_format = 'retina'

### 1. Load Train / Test Splits

In [ ]:
X_train = pd.read_csv('../data/X_train.csv')
X_test  = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv')
y_test  = pd.read_csv('../data/y_test.csv')

describe_dataset(X_train, X_test, y_train, y_test)

### 2. Class Balancing Strategies

Because the positive class is ~10 %, raw accuracy is a misleading metric. We experiment with:

| Strategy | Mechanism |
|----------|-----------|
| **Original** | Rely on `class_weight` parameter |
| **Undersampling** | Randomly remove majority-class rows until balanced |
| **Oversampling (SMOTE)** | Synthetically generate minority-class rows |

In [ ]:
X_train_under, y_train_under = undersample(X_train, y_train)
X_train_over,  y_train_over  = oversample(X_train, y_train)

print(f"Original    — X: {X_train.shape}, y: {y_train.shape}")
print(f"Undersampled — X: {X_train_under.shape}, y: {y_train_under.shape}")
print(f"Oversampled  — X: {X_train_over.shape},  y: {y_train_over.shape}")

### 3. Benchmark — Random Forest

We compare AUC-ROC as the primary metric (consistent with prior literature on this dataset). Hyperparameters are manually set here; tuning happens in Phase 3.

In [ ]:
rf_orig = RandomForestClassifier(max_depth=10, random_state=42, class_weight='balanced')
rf_orig.fit(X_train, y_train)
evaluate_model(rf_orig, X_test, y_test)

In [ ]:
rf_under = RandomForestClassifier(max_depth=10, random_state=42)
rf_under.fit(X_train_under, y_train_under)
evaluate_model(rf_under, X_test, y_test)

In [ ]:
rf_over = RandomForestClassifier(max_depth=10, random_state=42, class_weight={0: 1, 1: 4})
rf_over.fit(X_train_over, y_train_over)
evaluate_model(rf_over, X_test, y_test)

In [ ]:
compare_models(
    [rf_orig, rf_under, rf_over],
    ['RF — original', 'RF — undersampled', 'RF — oversampled (SMOTE)'],
    X_test, y_test
)

In [ ]:
_ = plot_feature_importance(rf_orig.feature_importances_, X_train.columns, top_n=20)

**Insight:** Undersampled RF and class-weighted RF on original data perform comparably. SMOTE oversampling slightly hurts performance, consistent with findings in the literature. `number_inpatient` and `discharge_disposition_id_*` dominate feature importance.

### 4. Benchmark — XGBoost

In [ ]:
xg_orig = xgb.XGBClassifier(random_state=42, learning_rate=0.001,
                             max_depth=6, scale_pos_weight=9)
xg_orig.fit(X_train, y_train)
evaluate_model(xg_orig, X_test, y_test)

In [ ]:
xg_under = xgb.XGBClassifier(random_state=42, learning_rate=0.01, max_depth=6)
xg_under.fit(X_train_under, y_train_under)
evaluate_model(xg_under, X_test, y_test)

In [ ]:
xg_over = xgb.XGBClassifier(random_state=42, learning_rate=0.001, scale_pos_weight=4)
xg_over.fit(X_train_over, y_train_over)
evaluate_model(xg_over, X_test, y_test)

In [ ]:
compare_models(
    [xg_orig, xg_under, xg_over],
    ['XGBoost — original', 'XGBoost — undersampled', 'XGBoost — oversampled'],
    X_test, y_test
)

In [ ]:
_ = plot_feature_importance(xg_orig.feature_importances_, X_train.columns, top_n=20)

**Insight:** XGBoost achieves slightly lower AUC than RF overall but is more sensitive to hyperparameter choices — there is likely headroom with tuning. The model leans heavily on a narrow set of features.

### 5. Benchmark — MLP (Multilayer Perceptron)

In [ ]:
_mlp_cfg = dict(hidden_layer_sizes=(128, 64), activation='relu',
                solver='adam', random_state=42)

mlp_orig  = MLPClassifier(**_mlp_cfg, max_iter=30)
mlp_orig.fit(X_train, y_train)
evaluate_model(mlp_orig, X_test, y_test)

In [ ]:
mlp_under = MLPClassifier(**_mlp_cfg, max_iter=20)
mlp_under.fit(X_train_under, y_train_under)
evaluate_model(mlp_under, X_test, y_test)

In [ ]:
mlp_over = MLPClassifier(**_mlp_cfg, max_iter=20)
mlp_over.fit(X_train_over, y_train_over)
evaluate_model(mlp_over, X_test, y_test)

In [ ]:
compare_models(
    [mlp_orig, mlp_under, mlp_over],
    ['MLP — original', 'MLP — undersampled', 'MLP — oversampled'],
    X_test, y_test
)

### 6. Ensemble Methods

All ensemble variants below are trained on **undersampled** data, the consistently best-performing configuration.

#### 6.1 Bagging

A Bagging classifier trains multiple base learners on random data subsets to reduce variance.

In [ ]:
clf_bag = BaggingClassifier(
    estimator=RandomForestClassifier(max_depth=10, random_state=42),
    random_state=42
)
clf_bag.fit(X_train_under, y_train_under)
evaluate_model(clf_bag, X_test, y_test)

#### 6.2 Soft Voting

Combines class probability estimates from RF, XGBoost, and MLP by averaging.

In [ ]:
clf_vote = VotingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(max_depth=10, random_state=42)),
        ('xgb', xgb.XGBClassifier(random_state=42, learning_rate=0.01, max_depth=6)),
        ('mlp', MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=20,
                              activation='relu', solver='adam', random_state=42)),
    ],
    voting='soft'
)
clf_vote.fit(X_train_under, y_train_under)
evaluate_model(clf_vote, X_test, y_test)

#### 6.3 Stacking

Uses base-learner predictions as meta-features for a Logistic Regression final estimator.

In [ ]:
clf_stack = StackingClassifier(
    estimators=[
        ('rf',  RandomForestClassifier(max_depth=10, random_state=42)),
        ('xgb', xgb.XGBClassifier(random_state=42, learning_rate=0.01, max_depth=6)),
        ('mlp', MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=20,
                              activation='relu', solver='adam', random_state=42)),
    ],
    final_estimator=LogisticRegression()
)
clf_stack.fit(X_train_under, y_train_under)
evaluate_model(clf_stack, X_test, y_test)

### 7. Full Model Comparison

In [ ]:
all_models  = [rf_orig, rf_under, rf_over,
               xg_orig, xg_under, xg_over,
               mlp_orig, mlp_under, mlp_over,
               clf_bag, clf_vote, clf_stack]

all_labels  = ['RF-orig', 'RF-under', 'RF-over',
               'XGB-orig', 'XGB-under', 'XGB-over',
               'MLP-orig', 'MLP-under', 'MLP-over',
               'Bagging', 'Voting', 'Stacking']

compare_models(all_models, all_labels, X_test, y_test)

### 8. Selection Decision

**Winner: Random Forest (undersampled)**

Rationale:
- Achieves the best or joint-best AUC-ROC
- Interpretable (feature importances)
- Faster to train than ensemble stacks
- Undersampling is simpler and more stable than SMOTE

This model proceeds to **Phase 3 — Evaluation, Feature Selection & Hyperparameter Tuning**.